In [1]:
from echo.settings import debug_mode
from echo.indexing import create_tables
from echo.runner import make_call
import nest_asyncio
import asyncio


nest_asyncio.apply()

seller = 'https://whatfix.com/'
buyer = 'https://www.manpowergroup.com'

create_tables(seller)


debug_mode

Created tables for https://whatfix.com/


False

In [2]:
from echo.step_templates.generic import CallType

call_id = 1
inputs = {
    'seller': seller,
    'call_id': call_id,
    'stakeholders': [
        "Product Manager",
        "Chief Financial Officer",
        "Chief Technology Officer",
        "VP of Sales"
    ]
}

discovery_data = asyncio.run(make_call(call_type=CallType.DISCOVERY.value, clients=[buyer], inputs=inputs))

Making discovery call for ['https://www.manpowergroup.com']
Getting analysis Data
Number of Clients:  1


Getting Data: 100%|██████████| 1/1 [00:00<00:00, 83.11it/s]

Getting Data for https://www.manpowergroup.com
Seller Research Data Found
Found Seller: https://whatfix.com/ Website Content
Buyer Research Data Found


In [3]:
inputs['call_id'] = call_id + 1
demo_data = asyncio.run(make_call(call_type=CallType.DEMO.value, clients=[buyer], inputs=inputs))

Making demo call for ['https://www.manpowergroup.com']
Getting analysis Data
Number of Clients:  1


Getting Data: 100%|██████████| 1/1 [00:00<00:00, 75.84it/s]

Getting Data for https://www.manpowergroup.com
dict_keys(['seller', 'call_id', 'stakeholders', 'buyer', 'call_type', 'demo_features', 'buyer_research', 'competitive_info', 'anticipated_qopcs', 'seller_website_content', 'buyer_website_content', 'previous_calls_analysis', 'seller_research', 'seller_pricing'])


In [4]:
inputs['call_id'] = call_id + 2
pricing_data = asyncio.run(make_call(call_type=CallType.PRICING.value, clients=[buyer], inputs=inputs))

Making pricing call for ['https://www.manpowergroup.com']
Getting analysis Data
Number of Clients:  1


Getting Data: 100%|██████████| 1/1 [00:00<00:00, 93.23it/s]

Getting Data for https://www.manpowergroup.com


In [5]:
inputs['call_id'] = call_id + 3
negotiation_data = asyncio.run(make_call(call_type=CallType.NEGOTIATION.value, clients=[buyer], inputs=inputs))

Making negotiation call for ['https://www.manpowergroup.com']
Getting analysis Data
Number of Clients:  1


Getting Data: 100%|██████████| 1/1 [00:00<00:00, 17.58it/s]

Getting Data for https://www.manpowergroup.com


In [ ]:
from echo.data.indexes import IndexType, IndexDataType
from echo.query_executor import Query, LlamaSubQuery, QueryChain
from echo.step_templates.utilities.account_plan_creation import QueryTypes


account_plan = Query(
    query="You're a strategic B2B seller. Based on the following public signals about {buyer}.\n" 
    "Extract 3-5 key initiatives or priorities the company is likely pursuing this year or quarter.\n"
    "Phrase each as a business goal. Do NOT include vague goals. Be specific.\n"
    "Signals for the various initiatives are given below:\n"

    "\nReturn format:\n"
    "- Initiative: clear description\n"
    "- Supporting evidence: source\n",
    sub_queries=[
        LlamaSubQuery(
            query="What are the top 3 financial priorities for the account to solve for?",
            index_type=IndexType.BUYER_ACCOUNT_PLAN,
            inputs={"query_type": QueryTypes.FMOD.value},
        ),
        LlamaSubQuery(
            query="What are the top 3 competitors for the account that that client needs to consider?",
            index_type=IndexType.BUYER_ACCOUNT_PLAN,
            inputs={"query_type": QueryTypes.COMPANALYSIS.value},
        ),
        LlamaSubQuery(
            query="What is the most relevant news for the account?",
            index_type=IndexType.BUYER_ACCOUNT_PLAN,
            inputs={"query_type": QueryTypes.RECENTNEWS.value},
        ),
        LlamaSubQuery(
            query="What are the top 3 strategic priorities for the account to solve for?",
            index_type=IndexType.BUYER_ACCOUNT_PLAN,
            inputs={"query_type": QueryTypes.STRATEGY.value},
        )
    ],
    output_name="account_plan"
)

account_plan_value_prop = Query(
    query="""
    You are a strategic sales assistant. Given a set of company initiatives and the product profile of the sellers product below as context,
    identify which initiatives are *relevant* to what this product solves.

    For each initiative:
    - Mark as Relevant or Not Relevant
    - If Relevant: explain which product capability maps to it
    - If Not Relevant: explain why it's not a fit (e.g., not adjacent, unrelated)


    output format:
        "initiative": "...",
        "relevant": not_relevant/ mid / highly relevant,
        "mapped_to_product": "...",
        "reasoning": "..."
        "similar buyers and their roi': "...",

    """,
    sub_queries=[
        LlamaSubQuery(
            query="What is the details on the industry and products of the buyer account?",
            index_type=IndexType.BUYER_RESEARCH.value,
        ),
        LlamaSubQuery(
            query="What are the top 3 financial priorities for the account to solve for?",
            index_type=IndexType.BUYER_ACCOUNT_PLAN,
            inputs={"query_type": QueryTypes.FMOD.value},
        ),
        LlamaSubQuery(
            query="What are the top 3 competitors that buyer might be worried about and want to tackle",
            index_type=IndexType.BUYER_ACCOUNT_PLAN,
            inputs={"query_type": QueryTypes.COMPANALYSIS.value},
        ),
        LlamaSubQuery(
            query="What is the most relevant news and recent media for the buyeraccount?",
            index_type=IndexType.BUYER_ACCOUNT_PLAN,
            inputs={"query_type": QueryTypes.RECENTNEWS.value},
        ),
        LlamaSubQuery(
            query="What are the top 3 strategic priorities for the account",
            index_type=IndexType.BUYER_ACCOUNT_PLAN,
            inputs={"query_type": QueryTypes.STRATEGY.value},
        ),
        LlamaSubQuery(
            query="What are the top value propositions of the sellers product and what pains do they solve for customers. Dont give generic answers, but deep pains and priotrities of their buyers theyve solved for",
            index_type=IndexType.SELLER_RESEARCH.value,
            inputs={"data_type": IndexDataType.SELLER_RESEARCH_DATA.value}
        ),
        LlamaSubQuery(
            query="What are the exhaustive use cases and benefits of the sellers product? Dont be generic, be specific and also include details of how the use cases are tackled by the sellers product",
            index_type=IndexType.SELLER_RESEARCH.value,
            inputs={"data_type": IndexDataType.SELLER_RESEARCH_DATA.value}
        ),
        LlamaSubQuery(
            query="What case studies and testimonials do we have for the sellers product? Please include details of the case studies and testimonials and how they align with the buyers priorities",
            index_type=IndexType.SELLER_RESEARCH.value,
            inputs={"data_type": IndexDataType.SELLER_RESEARCH_DATA.value},
        )
    ],
    output_name="account_plan_value_prop",
)

In [7]:
from echo.query_executor import aget_query_response, arun_query_chain
import asyncio

import nest_asyncio
nest_asyncio.apply()


inputs = {
    "seller": seller,
    "buyer": buyer,
}

In [ ]:
response = asyncio.run(aget_query_response(echo_query=account_plan, inputs=inputs))
print(response[0])

**ManpowerGroup Key Initiatives**

Based on the public signals about ManpowerGroup, here are 3-5 key initiatives or priorities the company is likely pursuing this year or quarter:

### Initiative 1: Bridge the Talent Gap
ManpowerGroup is likely prioritizing the development of innovative workforce solutions to address the critical talent shortage faced by organizations worldwide.

**Supporting Evidence:** [1](https://go.manpowergroup.com/talent-shortage) ManpowerGroup's 2024 Talent Shortage Survey reveals that 74% of employers globally struggle to find the skilled talent they need.

### Initiative 2: Expand Green Workforce Transformation
ManpowerGroup is likely focusing on expanding its green workforce transformation initiatives to help organizations navigate the transition to a sustainable economy.

**Supporting Evidence:** [2](https://go.manpowergroup.com/green-workforce-transformation) ManpowerGroup aims to develop, train, and place 10 million people in green jobs by 2030, underscori

In [ ]:
response = asyncio.run(aget_query_response(echo_query=account_plan_value_prop, inputs=inputs))
print(response[0])

Running sub query What is the details on the industry and products of the buyer account?
Sub query context {'query': 'What is the details on the industry and products of the buyer account?', 'context': "Relevant Context:\nThe buyer account belongs to a company in the Staffing and Recruiting industry, specifically an Enterprise-sized organization. The products and services offered by this buyer include a wide range of workforce solutions tailored to meet the needs of organizations and individuals in today's dynamic work environment. They focus on areas such as talent recruitment and management, workforce consulting and analytics, green workforce transformation, employee training and development, as well as sustainability and ESG reporting."}
Running sub query What are the top 3 financial priorities for the account to solve for?
Sub query context {'query': 'What are the top 3 financial priorities for the account to solve for?', 'context': "Relevant Context:\nThe top 3 financial prioritie

Overriding of current TracerProvider is not allowed


Sub query context {'query': 'What case studies and testimonials do we have for the sellers product? Please include details of the case studies and testimonials and how they align with the buyers priorities', 'context': "Relevant Context:\nThe seller's product has several use cases and testimonials that align with the buyers' priorities. The use cases include Employee Training, User Onboarding, Digital Transformation, Software Adoption, Hands-on Training, Simulation-based Training, User Behavior Analysis, Application Usage Analytics, and Software Optimization. These align with the buyers' priorities by focusing on enhancing employee training, user onboarding, driving digital transformation, software adoption, and optimizing software performance. The testimonials and case studies showcase how the seller's products have successfully addressed these priorities for other organizations, highlighting the effectiveness of the solutions provided."}
Running final query 
    You are a strategic s

In [13]:
competitor_differentiator_value_prop = Query(
    query="""You need to generate a few value propositions and business cases that the seller Whatfix can then use to sell their solution to the buyer Manpower group. 
            The seller is whatfix and the buyer is manpower group
            First identify the top financial, strategic, competitive and priorities evident from news and media to craft top issues and focus points of the buyer.
            Next deeply understand the sellers product, the core problems it has and does solve for its buyers.
            Please make sure to properly align value prop to actual business cases and not just generic value prop. 
            Also understand deeply what the seller sells and the kind of impact it can have before answering. 
            Think deeply
            Now finally, craft a set of value propositions and business cases that the sellers product can solve in alignment with the buyers priorities identified. This will be used by an account executive to pitch the product to the buyer and align with their priorities. so be clear, detailed and specific.
            Use the sellers product info, testimonials, broad initiatives theyve tackled for other customers and how they can align with the buyers strategic, financial and competitive priorities. Also include news and media about the buyer into consideration for further hints and signals on buyer priorities
            """,
    sub_queries=[
        LlamaSubQuery(
            query="What is the details on the industry and products of the buyer account?",
            index_type=IndexType.BUYER_RESEARCH.value,
            inputs={"data_type": IndexDataType.BUYER_RESEARCH_DATA.value}
        ),
        LlamaSubQuery(
            query="What are the top 3 financial priorities for the account to solve for?",
            index_type=IndexType.BUYER_ACCOUNT_PLAN,
            inputs={"query_type": QueryTypes.FMOD.value},
        ),
        LlamaSubQuery(
            query="What are the top 3 competitors that buyer might be worried about and want to tackle",
            index_type=IndexType.BUYER_ACCOUNT_PLAN,
            inputs={"query_type": QueryTypes.COMPANALYSIS.value},
        ),
        LlamaSubQuery(
            query="What is the most relevant news and recent media for the buyeraccount?",
            index_type=IndexType.BUYER_ACCOUNT_PLAN,
            inputs={"query_type": QueryTypes.RECENTNEWS.value},
        ),
        LlamaSubQuery(
            query="What are the top 3 strategic priorities for the account to solve for?",
            index_type=IndexType.BUYER_ACCOUNT_PLAN,
            inputs={"query_type": QueryTypes.STRATEGY.value},
        ),
        LlamaSubQuery(
            query="What are the top value propositions of the sellers product and what pains do they solve for customers. Dont give generic answers, but deep pains and priotrities of their buyers theyve solved for",
            index_type=IndexType.SELLER_RESEARCH.value,
            inputs={"data_type": IndexDataType.SELLER_RESEARCH_DATA.value}
        ),
        LlamaSubQuery(
            query="What are the exhaustive use cases and benefits of the sellers product? Dont be generic, be specific and also include details of how the use cases are tackled by the sellers product",
            index_type=IndexType.SELLER_RESEARCH.value,
            inputs={"data_type": IndexDataType.SELLER_RESEARCH_DATA.value}
        ),
        LlamaSubQuery(
            query="What case studies and testimonials do we have for the sellers product? Please include details of the case studies and testimonials and how they align with the buyers priorities",
            index_type=IndexType.SELLER_RESEARCH.value,
            inputs={"data_type": IndexDataType.SELLER_RESEARCH_DATA.value}
        )
    ],
)

In [14]:
response = asyncio.run(aget_query_response(echo_query=competitor_differentiator_value_prop, inputs=inputs))
print(response[0])

Running sub query What is the details on the industry and products of the buyer account?
Sub query context {'query': 'What is the details on the industry and products of the buyer account?', 'context': 'Relevant Context:\nThe buyer operates in the Staffing and Recruiting industry and offers a wide range of services tailored to meet the needs of organizations and individuals in today’s dynamic work environment. Their products include talent recruitment and management, workforce consulting and analytics, green workforce transformation, employee training and development, as well as sustainability and ESG reporting.'}
Running sub query What are the top 3 financial priorities for the account to solve for?
Sub query context {'query': 'What are the top 3 financial priorities for the account to solve for?', 'context': "Relevant Context:\nThe top 3 financial priorities for the account to solve for would be understanding the company's revenue, assessing its financial size within the industry, an

Overriding of current TracerProvider is not allowed


Sub query context {'query': 'What case studies and testimonials do we have for the sellers product? Please include details of the case studies and testimonials and how they align with the buyers priorities', 'context': "Relevant Context:\nThe seller's product has several use cases and testimonials that align with the buyers' priorities. The use cases include Employee Training, User Onboarding, Digital Transformation, Software Adoption, Hands-on Training, Simulation-based Training, User Behavior Analysis, Application Usage Analytics, and Software Optimization. These align with the buyers' priorities by focusing on enhancing employee training, user onboarding, software adoption, change management, and optimizing software performance. The testimonials and case studies showcase how the seller's products have successfully addressed these priorities for other organizations, highlighting the effectiveness of the solutions provided."}
Running final query You need to generate a few value propos

In [8]:
# 1) multi threading team gen 
multi_threading_team_gen = Query(
    query="""
        You're an experienced enterprise seller. Given these company initiatives, 
        for the ones marked relevant to the seller's product,
        infer which internal team likely owns or sponsors each initiative. 
        If multiple teams are involved, note primary and secondary.

        Input:
        {account_plan}

        Return format:
        - Initiative: ...
        - Likely owning team(s): ...
        - Reasoning:
    """,
    sub_queries=[
        LlamaSubQuery(
            query="What is the details on the industry and products of the buyer account?",
            index_type=IndexType.BUYER_RESEARCH.value,
            inputs={"data_type": IndexDataType.BUYER_RESEARCH_DATA.value}
        )
    ],
    output_name="multi_threading_team_gen",
)

# 2) do a perplexica serach here using team name and buyer name FOR EACH INITIATIVE RETURNED FROM ABOVE
# Search for all possible employees and leaders  from linkedin who belong to that team and extract role, name, and background


# 3) multi threading ROLE AND PERSON EXTRACTOR - use above response also aas input below additionally 

multi_threading_person_extractor = Query(
    query="""
        You are a strategic sales assistant. Given the list of initiatives, owning team and reasoning 
        for every relavant initiative the company is pursuing,
        
        Do the following:


        Given the company {buyer}, and the owning team of that initiative and the team members crawled from linkedin,
        return people who match titles commonly associated with owning this initiative.
        Focus on seniority, team fit, and tenure. Prioritize those with likely budget/influence.

        Also, Classify each as a champion, decision maker, gatekeeper and influncer within the team responsible for the initiative.
        champion - one who directly owns the pain and will want it solved
        decision maker- the one with power to purchase in the team and for the initiative
        gatekeeper - the one who will block the deal from happening or be tough to convince. This is the only role that could be outside the team like procurement , legal etc.
        influencer - the one who will influence the decision maker and champion to buy the product.

        Return:
        - initiative
        - Name
        - Title
        - Tenure
        - Team
        - Reason they likely own this initiative
        - Classification (champion, decision maker, gatekeeper, influencer) and why

        here is the list of initiatives and the owning team for each of them:
        {multi_threading_team_gen}
    """,
    sub_queries=[
        LlamaSubQuery(
            query="What is the details on the industry and products of the buyer account?",
            index_type=IndexType.BUYER_RESEARCH.value,
            inputs={"data_type": IndexDataType.BUYER_RESEARCH_DATA.value} ## Has demo data separately
        )
    ],
    output_name="multi_threading_person_extractor"
)


# 4) multi threading outreach generator

multi_threading_outreach_generator = Query(
    query="""
        You're a strategic AE selling {seller}. 
        You are given a list of initiatives, persona to target, title, reasoning and initiative they are participating in. For each buyer in the list
        Do the following:


        Based on this buyer’s title, initiative, 
        and recent activity, seller's product details and generate a 1st outreach email that aligns to their business goals and personal context.
        You are also given similar companies the seller has helped before below.

        Tone: Crisp, consultative, relevant.

        Return:
        - Subject line
        - Message body (under 100 words)
        - CTA
        - Persona
        - Title
        - Reasoning for message


        Input:
        {multi_threading_person_extractor}
        
    """,
    sub_queries=[
        LlamaSubQuery(
            query="what is the seller's product and what pains does it solve?",
            index_type=IndexType.SELLER_RESEARCH.value,
            inputs={"data_type": IndexDataType.SELLER_RESEARCH_DATA.value}
        ),
        LlamaSubQuery(
            query="what are some companies the sellers product has helped before? be specific and metric driven",
            index_type=IndexType.SELLER_RESEARCH.value,
            inputs={"data_type": IndexDataType.SELLER_RESEARCH_DATA.value}
        )
    ],
    output_name="multi_threading_outreach_generator"
)


In [9]:
query_chain = QueryChain(
    queries=[
        account_plan,
        multi_threading_team_gen,
        multi_threading_person_extractor,
        multi_threading_outreach_generator
    ]
)

In [10]:
response = asyncio.run(arun_query_chain(query_chain=query_chain, inputs=inputs))

Running query You're a strategic B2B seller. Based on the following public signals about {buyer}.
Extract 3-5 key initiatives or priorities the company is likely pursuing this year or quarter.
Phrase each as a business goal. Do NOT include vague goals. Be specific.
Signals for the various initiatives are given below:

Return format:
- Initiative: clear description
- Supporting evidence: source

Running sub query What are the top 3 financial priorities for the account to solve for?
Sub query context {'query': 'What are the top 3 financial priorities for the account to solve for?', 'context': "Relevant Context:\nThe top 3 financial priorities for the account to solve for would likely involve analyzing and managing the company's revenue growth strategies, assessing and optimizing its financial performance and profitability metrics, and evaluating its cost management and efficiency measures to ensure sustainable financial health and success."}
Running sub query What are the top 3 competito

Overriding of current TracerProvider is not allowed


Sub query context {'query': 'What is the details on the industry and products of the buyer account?', 'context': 'Relevant Context:\nThe buyer operates in the Staffing and Recruiting industry and offers a wide range of services tailored to meet the needs of organizations and individuals in today’s dynamic work environment. Their products include talent recruitment and management, workforce consulting and analytics, green workforce transformation, employee training and development, as well as sustainability and ESG reporting solutions.'}
Running final query 
        You're an experienced enterprise seller. Given these company initiatives, 
        for the ones marked relevant to the seller's product,
        infer which internal team likely owns or sponsors each initiative. 
        If multiple teams are involved, note primary and secondary.

        Input:
        **Key Initiatives and Priorities for ManpowerGroup**

Based on the public signals about ManpowerGroup, the following are 3-

Overriding of current TracerProvider is not allowed


Sub query context {'query': 'What is the details on the industry and products of the buyer account?', 'context': 'Relevant Context:\nThe buyer operates in the Staffing and Recruiting industry and offers a wide range of services tailored to meet the needs of organizations and individuals in today’s dynamic work environment. Their products include talent recruitment and management, workforce consulting and analytics, green workforce transformation, employee training and development, as well as sustainability and ESG reporting solutions.'}
Running final query 
        You are a strategic sales assistant. Given the list of initiatives, owning team and reasoning 
        for every relavant initiative the company is pursuing,
        
        Do the following:


        Given the company {buyer}, and the owning team of that initiative and the team members crawled from linkedin,
        return people who match titles commonly associated with owning this initiative.
        Focus on seniority,

Overriding of current TracerProvider is not allowed


Sub query context {'query': 'what are some companies the sellers product has helped before? be specific and metric driven', 'context': "Relevant Context:\nThe seller's product has helped companies like those in the technology sector by providing solutions for user behavior analysis, application usage analytics, and software optimization. The product has specifically assisted in enhancing user engagement and optimizing software performance through detailed insights gathered from no-code event tracking."}
Running final query 
        You're a strategic AE selling {seller}. 
        You are given a list of initiatives, persona to target, title, reasoning and initiative they are participating in. For each buyer in the list
        Do the following:


        Based on this buyer’s title, initiative, 
        and recent activity, seller's product details and generate a 1st outreach email that aligns to their business goals and personal context.
        You are also given similar companies th

TypeError: string indices must be integers, not 'str'